# Overview of Sub-Hourly Radar-Precipitation Datasets

This notebook gives a quick overview of all sub-hourly datasets in the `precipitation` catalog.

It creates two figures:
1. A spatial overview at one timestep that exists in all selected datasets, with a red bounding box for each dataset domain.
2. A timeline plot of each dataset's temporal coverage.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import cartopy.crs as ccrs

import mlcast_datasets

In [ ]:
cat = mlcast_datasets.open_catalog()
source_names = list(cat.precipitation)
source_names

In [ ]:
def infer_time_step_minutes(ds, sample_size=128):
    sample = pd.DatetimeIndex(ds["time"].isel(time=slice(0, sample_size)).values)
    if len(sample) < 2:
        return np.nan

    deltas = sample.to_series().diff().dropna()
    if deltas.empty:
        return np.nan

    return deltas.median().total_seconds() / 60.0


def select_plot_variable(ds):
    for var_name, da in ds.data_vars.items():
        if "time" in da.dims and da.ndim >= 3:
            return var_name
    raise ValueError("Could not find a 2D spatial variable with a time dimension.")


datasets = {}
summary = []

for name in source_names:
    ds = cat.precipitation[name].to_dask()
    step_min = infer_time_step_minutes(ds)
    datasets[name] = ds

    t = pd.DatetimeIndex(ds["time"].values)
    summary.append(
        {
            "dataset": name,
            "time_step_minutes": step_min,
            "start": t.min(),
            "end": t.max(),
            "n_times": len(t),
        }
    )

summary_df = pd.DataFrame(summary).sort_values("dataset").reset_index(drop=True)
summary_df

In [ ]:
subhourly = summary_df[summary_df["time_step_minutes"] < 60].copy()
subhourly_names = subhourly["dataset"].tolist()

if not subhourly_names:
    raise ValueError("No sub-hourly datasets found in cat.precipitation.")

subhourly

In [ ]:
time_indexes = {
    name: pd.DatetimeIndex(datasets[name]["time"].values) for name in subhourly_names
}

common_times = None
for name in subhourly_names:
    if common_times is None:
        common_times = time_indexes[name]
    else:
        common_times = common_times.intersection(time_indexes[name])

if common_times is None or len(common_times) == 0:
    raise ValueError("No common timestamp found across all sub-hourly datasets.")

common_time = common_times[-1]
common_time

In [ ]:
def get_data_crs(ds, var_name):
    grid_mapping_name = ds[var_name].attrs.get("grid_mapping")
    if grid_mapping_name and grid_mapping_name in ds:
        crs_wkt = ds[grid_mapping_name].attrs.get("crs_wkt")
        if crs_wkt:
            return ccrs.Projection(crs_wkt)
    return ccrs.PlateCarree()


def get_domain_bounds(da):
    spatial_dims = [d for d in da.dims if d != "time"]
    if len(spatial_dims) != 2:
        raise ValueError(f"Expected two spatial dims, got {spatial_dims}")

    y_dim, x_dim = spatial_dims
    x = da.coords.get(x_dim, da[x_dim])
    y = da.coords.get(y_dim, da[y_dim])

    if x.ndim != 1 or y.ndim != 1:
        raise ValueError(
            "Expected 1D coordinates for spatial dims; this helper currently supports rectilinear grids."
        )

    xmin = float(x.min().values)
    xmax = float(x.max().values)
    ymin = float(y.min().values)
    ymax = float(y.max().values)
    return xmin, xmax, ymin, ymax

## 1) Spatial overview at one common timestep

In [ ]:
# create equal area projection over Europe for the plot
crs = ccrs.LambertAzimuthalEqualArea(central_longitude=10, central_latitude=50)
fig, ax = plt.subplots(
    1,
    1,
    figsize=(12, 8),
    subplot_kw={"projection": crs},
)

palette = plt.cm.tab10(np.linspace(0, 1, len(subhourly_names)))
legend_handles = []

for i, name in enumerate(subhourly_names):
    ds = datasets[name]
    var_name = select_plot_variable(ds)
    data_crs = get_data_crs(ds, var_name)

    # Use first timestep to define the dataset spatial domain via non-NaN points.
    da0 = ds[var_name].isel(time=0)
    domain_mask = da0.notnull().astype(float).where(da0.notnull())
    # print number of valid points in domain_mask
    print(f"{name}: {domain_mask.sum().values} valid points in domain mask")

    color = palette[i]
    spatial_dims = [d for d in domain_mask.dims if d != "time"]
    y_dim, x_dim = spatial_dims
    x = domain_mask[x_dim].values
    y = domain_mask[y_dim].values
    z = domain_mask.values

    ax.contourf(
        x,
        y,
        z,
        levels=[0.5, 1.5],
        colors=[to_rgba(color, alpha=0.35)],
        transform=data_crs,
        antialiased=True,
    )

    legend_handles.append(
        Patch(facecolor=to_rgba(color, alpha=0.45), edgecolor="none", label=name)
    )

ax.coastlines(resolution="50m", color="black", linewidth=0.8)
ax.gridlines(draw_labels=["left", "bottom"], linestyle=":", alpha=0.5)
ax.set_title(
    f"Sub-hourly radar-precipitation dataset domains (mask from first timestep)"
)
ax.legend(handles=legend_handles, title="Datasets", loc="upper right")
# set extent to cover Europe
ax.set_extent([-25, 45, 30, 75], crs=ccrs.PlateCarree())
plt.tight_layout()

## 2) Temporal extent overview

In [ ]:
fig, ax = plt.subplots(figsize=(12, 1.4 * len(subhourly_names) + 2))

starts = []
ends = []

for i, name in enumerate(subhourly_names):
    t = time_indexes[name]
    start = t.min()
    end = t.max()
    starts.append(start)
    ends.append(end)

    ax.hlines(i, start, end, linewidth=8, alpha=0.9)
    ax.plot([start, end], [i, i], "|", color="black", markersize=12)

overlap_start = max(starts)
overlap_end = min(ends)
if overlap_start <= overlap_end:
    ax.axvspan(
        overlap_start,
        overlap_end,
        color="0.85",
        alpha=0.5,
        label="Common coverage window",
    )

ax.set_yticks(range(len(subhourly_names)))
ax.set_yticklabels(subhourly_names)
ax.set_xlabel("Time")
ax.set_ylabel("Dataset")
ax.set_title("Temporal extent of sub-hourly radar-precipitation datasets")
ax.grid(axis="x", linestyle=":", alpha=0.5)
if overlap_start <= overlap_end:
    ax.legend(loc="upper left")

plt.tight_layout()